In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
!pip install sktime -q
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from baselines_utils import (
    SequenceTensorExtractor,
    competition_scorer,
    evaluate_holdout
)
from ensemble_utils import HierarchicalBFRBEnsemble

E0000 00:00:1781246164.378934      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781246164.442813      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781246164.964047      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781246164.964111      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781246164.964114      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781246164.964117      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
TRAIN_ENSEMBLE = True
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "bayesian"  # "grid" or "bayesian"
random_state = 42
n_splits = 3
n_iter = 10
train_size = 0.4
error_score_constant = 0.0
verbose = 3
do_cross_val = False

full_wham = True  # Set to True for an 80/20 split, False for the default 20/18 split

if full_wham:
    test_size = 1 - train_size
else:
    test_size = min(0.25, 1 - train_size)  # Your original safe fallback logic

cv_object = GroupKFold(n_splits=n_splits) if do_cross_val else GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness & Upside-down corrections
if "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    train_df = train_df.drop(columns=["handedness"])

upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, 
    train_pct=train_size, 
    test_pct=test_size, 
    random_state=random_state
)

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

# 🚨 CRITICAL: y must contain sequence_id, is_target, orientation, and the target_col
y_train = train_sample_df[["sequence_id", "is_target", orientation_col, target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", orientation_col, target_col]].copy()
groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique()}")

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Train: 2910 seqs | 35.7%
Test:  2881 seqs  | 35.3%
Train sequences: 2910 | Test sequences: 2881


In [3]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    bayes_extractor_space = {
        "extractor__acc_modes": Categorical(["raw", "smoothed|velocity|jerk"]),
        "extractor__rotation_modes": Categorical([ "quaternion|angular_velocity|euler", "raw"]),
        "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats"]),
        "extractor__thm_modes": Categorical(["centered_diff"]),
        "extractor__sampling_rate": Integer(20, 200),
        "extractor__maxlen": Integer(20, 200),
        "extractor__window_size": Integer(10, 50),
        # "extractor__clip_value": Real(30.0, 150.0, prior="linear"),
        "extractor__interp_mode": Categorical(["linear", None]),
        "extractor__motion_filter_mode": Categorical([ "kalman", 'extended_kalman', None]),
        # "extractor__use_dead_reckoning": Categorical([True]),
        # "extractor__dead_reckoning_detrend": Categorical([True]),
        # "extractor__kalman_process_noise": Real(1e-6, 1e-1, prior="log-uniform"),
        # "extractor__kalman_measurement_noise": Real(1e-3, 1e2, prior="log-uniform"),
    }

    ensemble_param_space = {
        **bayes_extractor_space,
        # Layer 1: Binary BFRB
        "classifier__l1_num_kernels": Integer(1000, 2000),
        # "classifier__l1_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l1_class_weight": Categorical(["balanced", None]),
        # "classifier__l1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        # Layer 2: Orientation
        "classifier__l2_num_kernels": Integer(1000, 2000),
        # "classifier__l2_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l2_class_weight": Categorical(["balanced", None]),
        # "classifier__l2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        # Layer 3: BFRB per Orientation (Models 1-4)
        "classifier__l3_1_num_kernels": Integer(1000, 2000),
        # "classifier__l3_1_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l3_1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        "classifier__l3_2_num_kernels": Integer(1000, 2000),
        # "classifier__l3_2_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l3_2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        "classifier__l3_3_num_kernels": Integer(1000, 2000),
        # "classifier__l3_3_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l3_3_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        "classifier__l3_4_num_kernels": Integer(1000, 2000),
        # "classifier__l3_4_alpha": Real(1e1, 1e4, prior="log-uniform"),
        # "classifier__l3_4_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
        
        # "classifier__l3_class_weight": Categorical(["balanced", None]),
    }

else:  # GRID SEARCH
    grid_extractor_space = {
        "extractor__acc_modes": ["smoothed|velocity|displacement|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity|euler"], # FIXED from "raw"
        "extractor__tof_modes": ["pooled_stats|sensor_stats"],                     # FIXED from "raw"
        "extractor__thm_modes": ["centered_diff|diff"],                              # FIXED typo from "raww"
        "extractor__sampling_rate": [200],
        "extractor__maxlen": [160],
        "extractor__window_size": [40],
        "extractor__clip_value": [None],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": [None],
        "extractor__use_dead_reckoning": [False],
        "extractor__dead_reckoning_detrend": [False],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
    }

    ensemble_param_space = {
        **grid_extractor_space,
        # Layer 1
        "classifier__l1_num_kernels": [2000],
        "classifier__l1_alpha": [1e3],
        "classifier__l1_class_weight": ["balanced"],
        "classifier__l1_feature_selection_percentile": [50], # ADDED
        
        # Layer 2
        "classifier__l2_num_kernels": [2000],
        "classifier__l2_alpha": [1e4],
        "classifier__l2_class_weight": ["balanced"],
        "classifier__l2_feature_selection_percentile": [50], # ADDED
        
        # Layer 3
        "classifier__l3_1_num_kernels": [2000],
        "classifier__l3_1_alpha": [1e4],
        "classifier__l3_1_feature_selection_percentile": [50], # ADDED
        
        "classifier__l3_2_num_kernels": [2000],
        "classifier__l3_2_alpha": [1e4],
        "classifier__l3_2_feature_selection_percentile": [50], # ADDED
        
        "classifier__l3_3_num_kernels": [2000],
        "classifier__l3_3_alpha": [1e4],
        "classifier__l3_3_feature_selection_percentile": [50], # ADDED
        
        "classifier__l3_4_num_kernels": [2000],
        "classifier__l3_4_alpha": [1e4],
        "classifier__l3_4_feature_selection_percentile": [50], # ADDED
        
        "classifier__l3_class_weight": ["balanced"],
    }

In [4]:
# ============================================================
# MODEL TRAINING & EVALUATION LOOP
# ============================================================
results_list = []
fitted_models = {}

if TRAIN_ENSEMBLE:
    print("\n--- Training: Hierarchical BFRB Ensemble ---")
    
    # Optional: Manual dictionary override for Layer 3 models
    # Example: l3_override = {"Seated Straight": {"num_kernels": 2000, "alpha": 0.5}}
    l3_override_dict = None 

    ensemble_pipe = Pipeline([
        ("extractor", SequenceTensorExtractor(
            acc_modes="smoothed|velocity|jerk",
            rotation_modes="quaternion|angular_velocity",
            tof_modes="pooled_stats",
            thm_modes="centered_diff",
            sampling_rate=50,
            maxlen=150,
            window_size=40,
            clip_value=50.0,
            interp_mode="linear",
            motion_filter_mode="kalman",
            use_dead_reckoning=False,
            dead_reckoning_detrend=False,
            kalman_process_noise=1e-3,
            kalman_measurement_noise=1e-1
        )),
        ("classifier", HierarchicalBFRBEnsemble(
            orientation_col=orientation_col,
            target_col=target_col,
            l3_params_dict=l3_override_dict,
            random_state=random_state
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        ensemble_search = BayesSearchCV(
            ensemble_pipe, ensemble_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        ensemble_search = GridSearchCV(
            ensemble_pipe, ensemble_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    ensemble_search.fit(X_train, y_train, groups=groups)
    fitted_models["Hierarchical Ensemble"] = ensemble_search.best_estimator_
    
    y_pred_ens = ensemble_search.predict(X_test)
    
    eval_dict_ens = evaluate_holdout(y_test, y_pred_ens, target_col=target_col, verbose=True)
    score_ens = eval_dict_ens.get("competition_score", eval_dict_ens.get("holdout_score", 0))
    
    print(f"Hierarchical Ensemble Best CV Score: {ensemble_search.best_score_:.4f} | Holdout Score: {score_ens:.4f}")
    print(f"Best Params: {ensemble_search.best_params_}")
    
    results_list.append({
        "Model": "Hierarchical Ensemble",
        "CV Score": ensemble_search.best_score_,
        "Holdout Score": score_ens,
        "Best Params": ensemble_search.best_params_,
    })


--- Training: Hierarchical BFRB Ensemble ---
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__l1_num_kernels=1410, classifier__l2_num_kernels=1728, classifier__l3_1_num_kernels=1933, classifier__l3_2_num_kernels=1316, classifier__l3_3_num_kernels=1670, classifier__l3_4_num_kernels=1414, extractor__acc_modes=raw, extractor__interp_mode=None, extractor__maxlen=75, extractor__motion_filter_mode=extended_kalman, extractor__rotation_modes=raw, extractor__sampling_rate=43, extractor__thm_modes=centered_diff, extractor__tof_modes=pooled_stats|sensor_stats, extractor__window_size=13;, score=(train=1.000, test=0.604) total time= 4.1min
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__l1_num_kernels=1837, classifier__l2_num_kernels=1883, classifier__l3_1_num_kernels=1303, classifier__l3_2_num_kernels=1951, classifier__l3_3_num_kernels=1864, classifier__l3_4_num_kernels=1062, extractor__acc_modes=raw, extractor__interp_mode=li

In [5]:
# ============================================================
# FINAL SUMMARY
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "="*60)
print("BASELINES SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    print(f"\nDetailed holdout eval for best model: {best_name}")
    best_model = fitted_models[best_name]
    
    y_pred = best_model.predict(X_test)
    
    # Collapse y_test to sequence level to match the length of y_pred
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    print("\nClassification Report:")
    print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))


BASELINES SUMMARY
                Model  CV Score  Holdout Score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     Best Params
Hierarchical Ensemble  0.665628       0.706965 {'classifier__l1_num_kernels': 1812, 'classifier__l2_num_kernels': 1172, 'classifier__l3_1_num_kernels': 1598, 'classifier__l3_2_num_kernels': 1803, 'classifier__l3_3_num_kernels': 1523, 'classifier__l3_4_num_kernels': 1095, 'extractor__acc_modes': 'smoothed|velocity|jerk', 'ext